In [1]:
# -*- coding: utf-8 -*-
"""
Minh họa tối ưu theo quy tắc kinh nghiệm (Heuristic), sách Chương 19, mục 19.1.
Truy vấn: họ của nhân viên sinh sau 1957 làm dự án 'Aquarius'.
Chi phí = số dòng đọc từ các bảng + số dòng trung gian sinh ra.
Chạy: python demo_heuristic.py
"""
from datetime import date

# ----- DỮ LIỆU GIẢ LẬP (nhỏ để nhìn được) -----
EMPLOYEE = [
    {"Ssn": 1, "Lname": "An",   "Bdate": date(1955, 1, 9)},
    {"Ssn": 2, "Lname": "Binh", "Bdate": date(1960, 12, 8)},
    {"Ssn": 3, "Lname": "Chi",  "Bdate": date(1968, 7, 19)},
    {"Ssn": 4, "Lname": "Dung", "Bdate": date(1941, 6, 20)},
    {"Ssn": 5, "Lname": "Em",   "Bdate": date(1962, 9, 15)},
    {"Ssn": 6, "Lname": "Phuc", "Bdate": date(1972, 7, 31)},
    {"Ssn": 7, "Lname": "Giang", "Bdate": date(1969, 3, 29)},
    {"Ssn": 8, "Lname": "Hai",  "Bdate": date(1937, 11, 10)},
]
PROJECT = [
    {"Pnumber": 1, "Pname": "ProductX"},
    {"Pnumber": 2, "Pname": "ProductY"},
    {"Pnumber": 3, "Pname": "Aquarius"},
    {"Pnumber": 4, "Pname": "Computerization"},
]
WORKS_ON = [  # (nhân viên Essn làm dự án Pno)
    {"Essn": 1, "Pno": 1}, {"Essn": 1, "Pno": 2}, {"Essn": 2, "Pno": 1},
    {"Essn": 2, "Pno": 3}, {"Essn": 3, "Pno": 3}, {"Essn": 4, "Pno": 2},
    {"Essn": 5, "Pno": 3}, {"Essn": 6, "Pno": 4}, {"Essn": 7, "Pno": 3},
    {"Essn": 7, "Pno": 4}, {"Essn": 8, "Pno": 1}, {"Essn": 8, "Pno": 3},
]
MOC = date(1957, 12, 31)


def in_bang(ten, rows):
    print(f"\n{ten}: {len(rows)} dòng")
    if rows:
        cols = list(rows[0])
        print("   " + " | ".join(f"{c:<10}" for c in cols))
        for r in rows:
            print("   " + " | ".join(f"{str(r[c]):<10}" for c in cols))


def join(left, right, lkey, rkey):
    """Nối: ghép dòng của hai bảng có giá trị khóa bằng nhau."""
    return [{**l, **r} for l in left for r in right if l[lkey] == r[rkey]]


doc = len(EMPLOYEE) + len(WORKS_ON) + len(PROJECT)
print("=== DỮ LIỆU ===")
in_bang("EMPLOYEE", EMPLOYEE)
in_bang("WORKS_ON", WORKS_ON)
in_bang("PROJECT", PROJECT)

# ===================== CÁCH 1: CHƯA TỐI ƯU =====================
print("\n\n=== CÁCH 1: chưa tối ưu (ghép hết rồi mới lọc) ===")
t1 = [{**e, **w} for e in EMPLOYEE for w in WORKS_ON]
print(f"Bước 1: EMPLOYEE x WORKS_ON = {len(EMPLOYEE)} x {len(WORKS_ON)} = {len(t1)} dòng")
t2 = [{**a, **p} for a in t1 for p in PROJECT]
print(f"Bước 2: (kết quả trên) x PROJECT = {len(t1)} x {len(PROJECT)} = {len(t2)} dòng")
t3 = [r for r in t2 if r["Pnumber"] == r["Pno"] and r["Ssn"] == r["Essn"]
      and r["Pname"] == "Aquarius" and r["Bdate"] > MOC]
print(f"Bước 3: lọc theo điều kiện, còn {len(t3)} dòng")
chiphi1 = doc + len(t1) + len(t2) + len(t3)
print(f"Chi phí = {doc} (đọc 3 bảng) + {len(t1)} + {len(t2)} + {len(t3)} = {chiphi1}")
kq1 = sorted(r["Lname"] for r in t3)

# ===================== CÁCH 2: HEURISTIC =====================
print("\n\n=== CÁCH 2: heuristic (lọc trước, nối sau) ===")
p = [r for r in PROJECT if r["Pname"] == "Aquarius"]
in_bang("Bước 1: lọc PROJECT (Pname = 'Aquarius')", p)
e = [r for r in EMPLOYEE if r["Bdate"] > MOC]
in_bang("Bước 2: lọc EMPLOYEE (Bdate > 1957-12-31)", e)
j1 = join(p, WORKS_ON, "Pnumber", "Pno")
in_bang("Bước 3: nối dự án Aquarius với WORKS_ON", j1)
j2 = join(j1, e, "Essn", "Ssn")
in_bang("Bước 4: nối với nhân viên đã lọc", j2)
chiphi2 = doc + len(p) + len(e) + len(j1) + len(j2)
print(f"\nChi phí = {doc} (đọc 3 bảng) + {len(p)} + {len(e)} + {len(j1)} + {len(j2)} = {chiphi2}")
kq2 = sorted(r["Lname"] for r in j2)

# ===================== SO SÁNH =====================
print("\n\n=== SO SÁNH ===")
print("Kết quả cách 1:", kq1)
print("Kết quả cách 2:", kq2)
print("Giống nhau:", kq1 == kq2)
print(f"Chi phí: {chiphi1} (cách 1) so với {chiphi2} (cách 2), giảm {chiphi1 / chiphi2:.1f} lần")


=== DỮ LIỆU ===

EMPLOYEE: 8 dòng
   Ssn        | Lname      | Bdate     
   1          | An         | 1955-01-09
   2          | Binh       | 1960-12-08
   3          | Chi        | 1968-07-19
   4          | Dung       | 1941-06-20
   5          | Em         | 1962-09-15
   6          | Phuc       | 1972-07-31
   7          | Giang      | 1969-03-29
   8          | Hai        | 1937-11-10

WORKS_ON: 12 dòng
   Essn       | Pno       
   1          | 1         
   1          | 2         
   2          | 1         
   2          | 3         
   3          | 3         
   4          | 2         
   5          | 3         
   6          | 4         
   7          | 3         
   7          | 4         
   8          | 1         
   8          | 3         

PROJECT: 4 dòng
   Pnumber    | Pname     
   1          | ProductX  
   2          | ProductY  
   3          | Aquarius  
   4          | Computerization


=== CÁCH 1: chưa tối ưu (ghép hết rồi mới lọc) ===
Bước 1: EMPLOYEE x WORKS_O